# Consistency between windstorm tracking algorithms – Duration

Production date: 2026-10-DD.

Produced by: Enis Gerxhalija & Olivier Burggraaff (National Physical Laboratory).

## 🌍 Use case: C3S Enhanced Windstorm Service (EWS)

## ❓ Quality assessment question
* **How do the tracking algorithms included in the _Windstorm tracks and footprints_ dataset (Hodges, TempestExtremes) compare in terms of windstorm duration?**

As part of the Copernicus Climate Change Service's (C3S) Enhanced Windstorm Service (EWS),
the [_Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present_](https://doi.org/10.24381/bf1f06a9) dataset
is intended to help users in the insurance sector
better understand the risk caused by European windstorms or extra-tropical cyclones (ETCs).
ETCs are large low-pressure systems in mid-latitude regions associated with high winds and heavy rain [[Walker+20](https://doi.org/10.1002/wea.3861)].
These windstorms can cause severe socio-economic damages, with costs in the millions of euros and fatalities ranging from individuals to thousands [[Garnier+18](https://doi.org/10.1016/j.coastaleng.2017.06.014)].

To generate this dataset,
two windstorm tracking algorithms
–
_Hodges_/TRACK [[Hoskins+02](https://doi.org/10.1175/1520-0469(2002)059%3C1041:NPOTNH%3E2.0.CO;2), [Hodges+99](https://doi.org/10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2), [Hodges+95](https://doi.org/10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2)]
and
_TempestExtremes_ [[Ullrich+21](https://doi.org/10.5194/gmd-14-5023-2021)]
–
were applied to historical meteorological data from ECMWF's fifth-generation reanalysis ERA5 [[Hersbach+20](https://doi.org/10.1002/qj.3803)] to detect windstorm tracks.
The resulting dataset contains three products, namely
the windstorm tracks found by both algorithms,
the associated footprint of each windstorm,
and
yearly summary indicators for administrative regions.

Because the tracking algorithms have different approaches for detecting windstorms,
with
Hodges using 3-hourly 850 hPa relative vorticity
and
TempestExtremes using 6-hourly mean sea level pressure (MSLP),
they are intended to be complementary to each another.
ETCs are highly complex and diverse systems
and
different tracking algorithms have different sensitivities, strengths, and limitations [[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)].
For example,
the MSLP approach generally represents low-frequency, large-scale features,
while vorticity represents high-frequency, small-scale features
[[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1), [Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)]

Here,
we compare the outputs of both algorithms in terms of **duration**,
i.e.
when individual windstorm tracks are first/last detected and their total duration.
This comparison is performed as an internal consistency check,
rather than a validation using external data.
The aim is to provide guidance on the choice of algorithm and the consequences of using one over the other, or both at the same time.

## 📢 Quality assessment statement

```{admonition} These are the key outcomes of this assessment
:class: note

* More than 1000 more storms picked up by the Hodges tracking algorithm compared to the TempestExtremes (TE) algorithm since 1940 to present, with both trackers finding 4 and 7x more ETCs in the winter months (December, January, February) than the summer (June, July, August) respectively. The Hodges tracking algorithm finds 2.5x more storms in the winter than TE, and 1.5 more storms in the summer than TE.

* Using the track-to-track comparison method proposed by Hodges et al. (2003), track agreement was found to be 75% between the Hodges and TE tracking algorithms. Track agreement is found to be higher in the summer months (82%) compared to the winter months (71%).

* To evaluate the performance of both algorithms against known events, we analysed two well-documented transitioning extratropical cyclones, Ophelia (2017) and Oscar (2018), as well as the exceptional winter storm Klaus (2009). We found whilst both tracking algorithms find Ophelia (2017) with near-perfect agreement, Oscar was not found by Hodges and Klaus was not found by TE.

* Between the two tracking algorithms, TE finds more ETCs with longer lifetimes with the Hodges algorithm finding more ETCs with shorter lifetimes. Between matchup pairs the lifetime distributions are consistent with one another. 

* etc
```

## 📋 Methodology
This notebook examines the consistency in windstorm characteristics between the two tracking algorithms used in the [_Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present_](https://doi.org/10.24381/bf1f06a9) dataset.
Here,
we focus on the duration of windstorms as detected by each algorithm,
by investigating the individual windstorm tracks.

The analysis and results are organised in the following steps, which are detailed in the sections below:

**[](section-setup)**
* Import all required libraries.
* Define helper functions.

**[](section-duration)**
* Compare distribution of lifetimes for all windstorms between two algorithms.
* Compare distribution of lifetimes between matchups between two algorithms.

## 📈 Analysis and results

(section-setup)=
### 1. Code setup
```{note}
This notebook uses [earthkit](https://github.com/ecmwf/earthkit) for
downloading ([earthkit-data](https://github.com/ecmwf/earthkit-data))
and visualising ([earthkit-plots](https://github.com/ecmwf/earthkit-plots)) data.
Because earthkit is in active development, some functionality may change after this notebook is published.
If any part of the code stops functioning, please raise an issue on our GitHub repository so it can be fixed.
```

#### Import required libraries
In this section, we import all the relevant packages needed for running the notebook.

In [ ]:
# Input / Output
import earthkit.data as ekd

# General data handling
import calendar
from functools import partial, reduce
from operator import and_ as bitwise_and
import numpy as np
# np.seterr(divide="ignore")  # Ignore divide-by-zero warnings
import pandas as pd
import xarray as xr 

# Windstorm track matchups
import haversine as hs

# Visualisation
import earthkit.plots as ekp
from earthkit.plots.styles import Style
import matplotlib.pyplot as plt
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["figure.constrained_layout.use"] = True
plt.rcParams["figure.labelweight"] = "bold"
plt.rcParams["figure.titleweight"] = "bold"
plt.rcParams["legend.columnspacing"] = 1.5
plt.rcParams["legend.handletextpad"] = 0.5
from matplotlib.lines import Line2D
import cartopy
from cartopy import crs as ccrs
from IPython.display import HTML

# Visualisation in Jupyter book -- automatically ignored otherwise
try:
    from myst_nb import glue
except ImportError:
    glue = None

# Type hints
from typing import Any, Iterable, Optional
from earthkit.plots.geography.domains import Domain
AnyDomain = (Domain | str)
from pandas.io.formats.style import Styler as PandasStyler

#### Helper functions
This section defines some functions and variables used in the following analysis, allowing code cells in later sections to be shorter and ensuring consistency.

##### Data downloading & (pre-)processing

The following cell contains some pre-defined constants for convenience,
such as a list of variable names in the data:

In [ ]:
# Data
TRACKING_ALGORITHMS = ["hodges", "tempest_extremes"]

# Seasons
MONTHS              = calendar.month_name[1:]    # Named months of the year; [1:] drops an empty entry at the start
DJF                 = calendar.month_name[12:] + calendar.month_name[1:3]  # Northern Hemisphere winter: December-January-February
MAM                 = calendar.month_name[3:6]   # Northern Hemisphere spring: March-April-May
JJA                 = calendar.month_name[6:9]   # Northern Hemisphere summer: June-July-August
SON                 = calendar.month_name[9:12]  # Northern Hemisphere autumn: September-October-November
SEASONS             = {"DJF": DJF, "MAM": MAM, "JJA": JJA, "SON": SON,}
month2season        = {month: season  # Convert from month to season, e.g. "December": "DJF"
                       for season, months in SEASONS.items()
                       for month in months
                      }

The following cell provides some convenience functions for working with Pandas DataFrames,
such as manipulating their indices:

In [ ]:
# Helper function: Add dummy index level to existing DataFrame
# (used here to allow certain functions to be re-used with different data)
def add_dummy_index(df: pd.DataFrame, name: str, value: Any) -> pd.DataFrame:
    """ Add a new index level that contains just a dummy value. """
    return pd.concat({value: df}, names=[name])

##### Statistics

The following functions calculate and display statistics on windstorm track duration:

In [ ]:
# Track dates
def summarise_track_dates(data: pd.DataFrame) -> pd.DataFrame:
    """
    For each windstorm track in `data`, get the start/end dates and most important meteorological parameters.
    """
    # Sort by timestamp first
    data = data.sort_values(["algorithm", "id", "time"])

    # Summarise each track (algorithm-id combination)
    summary = tracks.groupby(["algorithm", "id"]).agg(  # Apply aggregation operations to each pair of algorithm-id values
                  time_start=("time", "first"),
                  time_end=("time", "last"),
                  latitude_start=("latitude", "first"),
                  latitude_end=("latitude", "last"),
                  longitude_start=("longitude", "first"),
                  longitude_end=("longitude", "last"),
                  fg10_max=("fg10", "max"),
                  msl_min=("msl", "min"),
              )

    return summary


# Track durations
def compute_track_durations(track_summary: pd.DataFrame) -> pd.DataFrame:
    """ Given a windstorm track summary, compute the duration [in days] of each track from the start/end dates. """
    # Make a copy to avoid overwriting the original
    data = track_summary.copy()

    # Remove columns that are not time / coordinate related
    keep_keys = ["time", "latitude", "longitude"]
    data = data[[col for col in data.columns if any(key in col for key in keep_keys)]]

    # Calculate duration in days
    data["duration"] = (data["time_end"] - data["time_start"]) / pd.Timedelta(days=1)

    # Reorder columns
    data = data[["time_start", "time_end", "duration", "latitude_start", "latitude_end", "longitude_start", "longitude_end"]]

    return data


# Summary statistics on track durations
def track_duration_statistics(track_summary: pd.DataFrame) -> pd.DataFrame:
    """
    Compute summary statistics of windstorm track duration, by season and by algorithm.
    """
    # Make a copy to avoid overwriting the original
    data = track_summary.copy()

    # Calculate overall statistics
    stats_all = data.groupby(["algorithm",])["duration"].describe()
    stats_all = add_dummy_index(stats_all, "season", "All")  # Add "season: All" index to prepare for concatenation later

    # Calculate statistics by season
    data["season"] = data["time_start"].dt.month_name().replace(month2season)  # Add season column
    stats_by_season = data.groupby(["season", "algorithm",])["duration"].describe()

    # Combine into one table
    stats = pd.concat([stats_all, stats_by_season])
    stats = stats.loc[["All", *SEASONS.keys()]]  # Order: All, DJF, ... SON

    # Adjust data types
    stats["count"] = stats["count"].astype(int)

    return stats


def display_track_duration_statistics(data: pd.DataFrame) -> PandasStyler:
    """
    Display track duration more cleanly.
    Show count as int, everything else as float with 2 decimals.
    Colour all rows by the first index for easy comparison.
    """
    # Setup: Colour by element on the first index (e.g. by season)
    first = data.index.get_level_values(0).unique()
    first_colours = {index_value: "#f2f2f2" if i % 2 == 0 else "#ffffff"
                     for i, index_value in enumerate(first)}

    def colour_row(row):
        colour = first_colours[row.name[0]]
        return [f"background-color: {colour}"] * len(row)

    def colour_index(index):
        return [f"background-color: {first_colours[index_value]}"
                for index_value in data.index.get_level_values(0)]

    styled = (
        stats.style
             .format(lambda x: f"{x:.2f}")                    # Show all columns with 2 decimals
             .format(lambda x: f"{x:.0f}", subset=["count"])  # Show count with 0 decimals (integer)
             .apply(colour_row, axis=1)  # Colour data  according to first index level
             .apply_index(               # Colour index according to first index level
                 colour_index,
                 axis=0, level=[0, 1],
             )
    )
    return styled

The following functions find and analyse track matchups following the method described in [[Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)]:

In [ ]:
# Match / Filter by time
def find_overlapping_timestamps(data: pd.DataFrame, *, algorithms: Iterable[str]=TRACKING_ALGORITHMS) -> pd.DataFrame:
    """ Find timestamps that occur in both elements of `algorithms`, e.g. "hodges" and "tempest_extremes". """
    # Split dataset by tracking algorithm
    data_split = [data.loc[alg].reset_index(level="id") for alg in algorithms]

    # Find overlapping timestamps between the two algorithms for each storm ID
    time_overlap = pd.merge(*data_split,  # * splats the individual elements of data_split
                            on="time", how="inner",  # Find all points where the "time" column matches between algorithms
                            suffixes=[f"_{alg}" for alg in algorithms]  # Suffix with _hodges, _tempest_extremes
                           )
    return time_overlap

def match_by_time(data: pd.DataFrame, *, algorithms: Iterable[str]=TRACKING_ALGORITHMS) -> pd.DataFrame:
    """ Identify matching tracks based on temporal overlap, and calculate how much they overlap (% of length). """
    # Count overlapping time points for each track pair
    time_overlap = find_overlapping_timestamps(data, algorithms=algorithms)

    overlap_stats = (
        time_overlap
        .groupby([f"id_{alg}" for alg in algorithms])["time"]  # Loop through all matched pairs of (id_hodges, id_tempest_extremes)
        .agg(  # Calculate aggregated statistics
            length_overlap=pd.Series.nunique,  # Count the number of unique timestamps in each matched pair
            time_start="first",                # Get the first overlapping timestamp
        )
    ).reset_index()  # Turn (id_hodges, id_tempest_extremes) into columns
    
    overlap_stats = overlap_stats.rename(columns={"time": "length_overlap"})  # Rename column for clarity
    overlap_stats.index = overlap_stats.index.rename("id_matchup")            # Rename index for clarity

    # For each matching pair, get the lengths of the corresponding tracks in Hodges and TempestExtremes
    def process_individual_match_by_time(row_overlap: pd.Series) -> pd.Series:
        """ For one match-up row, get the individual track lengths. """
        # Get the original tracks for this ID in both algorithms
        ids_by_algorithm = {alg: row_overlap[f"id_{alg}"] for alg in algorithms}
    
        # Count how many timestamps this track has in both algorithms
        individual_tracks = {alg: data.loc[alg, ID] for alg, ID in ids_by_algorithm.items()}
        individual_lengths = {f"length_{alg}": track["time"].nunique() for alg, track in individual_tracks.items()}
    
        # Bundle into Pandas Series and return
        result = pd.Series(individual_lengths)
        return result

    individual_lengths = overlap_stats.apply(process_individual_match_by_time, axis=1)
    
    # Convert to DataFrame and filter matches
    matches_in_time = pd.concat([overlap_stats, individual_lengths], axis=1)

    # Determine how much overlap there is, for filtering later
    matches_in_time["length_mean"] = matches_in_time[[f"length_{alg}" for alg in algorithms]].mean(axis=1)
    matches_in_time["overlap_fraction"] = matches_in_time["length_overlap"] / matches_in_time["length_mean"]
    
    return matches_in_time

def filter_by_time_overlap(data: pd.DataFrame, *,
                           overlap_col: str="overlap_fraction", threshold: float=0.6) -> pd.DataFrame:
    """ Return only those elements of `data` where `overlap_col` >= threshold. Default is 0.6, from Hodges+03. """
    return data[data[overlap_col] >= threshold]


# Match / Filter by distance
def match_by_mean_distance(data: pd.DataFrame, *, existing_matchups: Optional[pd.DataFrame]=None,
                           algorithms: Iterable[str]=TRACKING_ALGORITHMS) -> pd.DataFrame:
    """
    Identify matching tracks based on temporal overlap, and calculate the mean distance between matched-up timestamps.
    `existing_matchups` can be a DataFrame with known match-ups, e.g. those obtained from matching and filtering on timestamp overlap.
        If provided, only matchups in `existing_matchups` will be considered.
    """
    # Find overlapping timestamps (i.e. timestamps that occur in both algorithms)
    # and calculate the distance between the locations provided by each algorithm
    time_overlap = find_overlapping_timestamps(data, algorithms=algorithms)
    coordinates_by_algorithm = [[f"{coordinate}_{algorithm}"
                                 for coordinate in COORDINATES]
                                for algorithm in algorithms]
    time_overlap_by_algorithm = [time_overlap[coords] for coords in coordinates_by_algorithm]
    time_overlap["mean_distance"] = hs.haversine_vector(*time_overlap_by_algorithm, unit=hs.Unit.KILOMETERS)

    # Calculate the mean distance for each match-up, across all overlapping timestamps
    distances_overlap = time_overlap.groupby([f"id_{alg}" for alg in algorithms])[["mean_distance"]].mean()
    distances_overlap = distances_overlap.reset_index()  # Turn (id_hodges, id_tempest_extremes) into columns
    distances_overlap.index = distances_overlap.index.rename("id_matchup")  # Rename index for clarity

    # If there were existing matchups (e.g. based on time), perform an inner join
    # This will filter out any matchups in `distances_overlap` that are not present in `existing_matchups`
    # (e.g. those filtered out on not having enough overlapping timestamps)
    if existing_matchups is not None:
        distances_overlap = (
            existing_matchups
            .reset_index()  # Move id_matchup to a column
            .merge(distances_overlap,  # Perform the inner join
                   how="inner",  # Find all points where the "time" column matches between algorithms
                   on=[f"id_{alg}" for alg in algorithms],
             )
            .set_index("id_matchup")  # Restore Matchup ID index
        )

    # Select best matches only -- sort by distance (closest first) and pick the first element for every storm
    distances_overlap = distances_overlap.sort_values("mean_distance")
    for alg in algorithms:  # Drop duplicates in Hodges, then in TempestExtremes (if provided in that order)
        distances_overlap = distances_overlap.drop_duplicates(subset=[f"id_{alg}"], keep="first")

    # Return result
    return distances_overlap

def filter_by_mean_distance(data: pd.DataFrame, *,
                            overlap_col: str="mean_distance", threshold: float=300.) -> pd.DataFrame:
    """ Return only those elements of `data` where `overlap_col` < threshold. Default is 300 km, from Hodges+03. """
    return data[data[overlap_col] < threshold]


# Display matchups
def display_track_matchups(data: pd.DataFrame, *, length: int=10) -> pd.DataFrame:
    """
    Display a table of track matchups (as a DataFrame, no HTML styling).
    Removes unwanted columns and applies rounding.
    `length` determines how many elements (at the start + at the end) are shown.
    """
    # Make copy to prevent overwriting data
    display_df = data.copy()  

    # Remove unwanted columns
    display_df = display_df.drop(columns=["length_overlap", "length_mean"])  # Redundant with existing columns

    # Apply number formatting
    display_df["overlap_fraction"] = display_df["overlap_fraction"].map(lambda x: f"{x:.0%}")
    display_df["mean_distance"]    = display_df["mean_distance"].map(   lambda x: f"{x:.0f}")

    return display_df

##### Visualisation

The following cell defines
the styles used for the different algorithms in line plots:

In [ ]:
# General styles for consistency
# Colours come from Paul Tol's notes (https://sronpersonalpages.nl/~pault/)
ALGORITHM_COLOURS = {"hodges": "#997700", "tempest_extremes": "#6699CC",
                     "difference": "black", "ratio": "black", "matchup": "black",
                    }
ALGORITHM_MARKERS = {"hodges": "s", "tempest_extremes": "^",
                     "difference": "o", "ratio": "o", "matchup": "o",
                    }

The following cell contains some base helper functions (e.g. displaying in Jupyter Notebook or Jupyter Book style):

In [ ]:
# Visualisation: Helper functions, general
def _glue_or_show(fig: plt.Figure, glue_label: Optional[str]=None) -> None:
    """
    If `glue` is available, glue the figure using the provided label.
    If not, display the figure in the notebook.
    """
    try:
        glue(glue_label, fig, display=False)
    except TypeError:
        plt.show()
    finally:
        plt.close()

def _add_textbox_to_subplots(text: str, *axs: plt.Axes | ekp.Subplot, right=False) -> None:
    """ Add a text box to each of the specified subplots. """
    # Get the plt.Axes for each ekp.Subplot
    axs = [subplot.ax if isinstance(subplot, ekp.Subplot) else subplot for subplot in axs]

    # Set up location
    x = 0.95 if right else 0.05
    horizontalalignment = "right" if right else "left"

    # Add the text
    for ax in axs:
        ax.text(x, 0.95, text, transform=ax.transAxes,
        horizontalalignment=horizontalalignment, verticalalignment="top",
        bbox={"facecolor": "white", "edgecolor": "black", "boxstyle": "round",
              "alpha": 1})

The following functions are also base helper functions, but specific to geospatial plots.
Administrative boundaries are downloaded from
the _Geographic Information System of the Commission_ (GISCO)
through earthkit-plots,
and are © EuroGeographics.

In [ ]:
# Helper constants
DOMAIN_TRACKS = "North Atlantic"

# Visualisation: Helper functions for geospatial plots
def decorate_fig(fig: ekp.Figure, *, title: Optional[str]="") -> None:
    """ Decorate an earthkit figure with land, coastlines, etc. """
    fig.land()
    fig.borders(source="gisco", color="black")
    fig.coastlines(color="black")
    fig.gridlines(linestyle=plt.rcParams["grid.linestyle"])
    fig.title(title)

The following functions display geospatial comparisons of individual windstorm tracks:

In [ ]:
# Helper functions
def select_track_data_for_ids(track_data: pd.DataFrame, track_ids: dict[str, int]) -> dict[str, pd.DataFrame]:
    """ Get the tracks for each algorithm/ID pair in `track_ids` from the main table `track_data`. """
    # Find data
    tracks_selected = {alg: track_data.loc[alg, ID] for alg, ID in track_ids.items()}

    # Pre-process: Sort by time, reset index, and select variables
    tracks_selected = {alg: track.sort_values("time") for alg, track in tracks_selected.items()}
    tracks_selected = {alg: track.reset_index(drop=True).set_index("time") for alg, track in tracks_selected.items()}
    tracks_selected = {alg: track[["latitude", "longitude"]] for alg, track in tracks_selected.items()}

    return tracks_selected

def plot_track_geospatial(chart: ekp.Map, track: pd.DataFrame, algorithm: str, **kwargs) -> None:
    """
    Plot a windstorm track on a geospatial map, with scatter markers and a dashed line.
    Uses algorithm-specific colours and markers.
    """
    # Setup
    kwargs_shared = {"transform": ccrs.PlateCarree()} | kwargs
    
    # Plot individual points
    chart.ax.scatter(track["longitude"], track["latitude"],
                     color=ALGORITHM_COLOURS[algorithm], marker=ALGORITHM_MARKERS[algorithm],
                     edgecolors="black", s=50,
                     zorder=3,
                     **kwargs_shared)

    # Plot lines connecting points
    chart.ax.plot(track["longitude"], track["latitude"],
                  color=ALGORITHM_COLOURS[algorithm],
                  linestyle="--", linewidth=3,
                  zorder=3-0.1, alpha=0.7,
                  **kwargs_shared)

# Main function
def compare_tracks_geospatial(track_data: pd.DataFrame, track_ids: dict[str, int] | Iterable[dict[str, int]],
                              glue_label: Optional[str]=None) -> None:
    """
    Plot a geospatial comparison between storm tracks in `track_data` from two different tracking algorithms.
    `track_ids` is a Dictionary of track IDs, e.g. {'id_hodges': 1702, 'id_tempest_extremes': 787,}.
    """
    # Check if multiple track_id pairs are being provided or only one
    if isinstance(track_ids, dict):  # Single pair
        track_ids = [track_ids,]  # Create list of length 1, to iterate over

    # Create map
    n_pairs = len(track_ids)
    n_rows = n_pairs // 2 + n_pairs % 2  # One row for every 2 (= n_cols) track_id pairs, plus the remainder (+1 for odd numbers)
    fig = ekp.Figure(rows=n_rows, columns=2, figsize=(10, 1+3.1*n_rows))
    # (figsize is tailored for a 2x2 plot, may not look good for other combinations)

    # Loop over each pair and plot them in their own panel
    for id_pair in track_ids:
        # Remove "id_" prefix
        id_pair = {key.removeprefix("id_"): val for key, val in id_pair.items()}

        # Select desired tracks only
        id_pair = {key: val for key, val in id_pair.items() if val is not None}  # Filter out `None` values (dummies)
        tracks_selected = select_track_data_for_ids(track_data, id_pair)

        # Plot tracks
        chart = fig.add_map(domain=DOMAIN_TRACKS)
        for algorithm, track in tracks_selected.items():
            plot_track_geospatial(chart, track, algorithm)

        # Legend for algorithm lines
        handles = [Line2D([0], [0],
                          color=ALGORITHM_COLOURS[alg], linestyle="--", linewidth=3,
                          marker=ALGORITHM_MARKERS[alg], markersize=8, markerfacecolor=ALGORITHM_COLOURS[alg], markeredgecolor="black",
                          label=alg,)
                   for alg in tracks_selected]
        chart.ax.legend(handles=handles, loc="upper left")

        # Title
        title_ids = " – ".join(f"{alg} ID {ID}" for alg, ID in id_pair.items())
        chart.title(title_ids)

    # Decorate figure
    decorate_fig(fig, title="Storm track comparison")

    # Show result
    _glue_or_show(fig.fig, glue_label)

The following functions display comparisons of windstorm track duration:

In [ ]:
def plot_lifetime_distribution(
    data,
    labels,
    colors,
    bins=np.arange(0, 50, 1),
    sigma=1
):

    """
    Plot probability distributions of storm lifetimes.
    """

    plt.figure()

    count = 0 
    
    for df, label in zip(data, labels):

        df = df.copy()

        # Lifetime (hours)
        lifetime = (df["end_time"] - df["start_time"]).dt.total_seconds() / (60*60)

        # Histogram
        hist, bin_edges = np.histogram(lifetime, bins=bins, density=True)

        # Bin centres
        centres = (bin_edges[:-1] + bin_edges[1:]) / 2

        # Plot
        plt.plot(centres, hist, label=label, color = colors[count])

        count += 1 
        
    # Labels and styling
    plt.xlabel("Lifetime (hours)")
    plt.ylabel("Probability density")
    plt.title("Cyclone Lifetime Distribution")
    plt.grid(True)
    plt.legend()

    plt.show()

(section-duration)=
### 2. Windstorm duration
We compare the windstorm tracks found by the Hodges and TempestExtremes algorithms.
The following cell defines the years for which we will download data;
by default, this is 1940–2025, but these years can be modified below when running the notebook yourself:

In [ ]:
# Customisation
# Modify the range below to change the years considered
year_start, year_end = 1940, 2025

#### Download windstorm tracks
We download the windstorm tracks from the
[sis-european-wind-storm-reanalysis](https://doi.org/10.24381/bf1f06a9)
CDS entry.
This is a simple table which can be loaded into a Pandas [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).
For each algorithm (Hodges, TempestExtremes),
this table contains all timestamps with corresponding location (latitude, longitude) and some fields describing physical characteristics at that timestamp and location (see the [Product User Guide](https://confluence.ecmwf.int/x/jHaXGw#WindstormtracksandfootprintsderivedfromreanalysisoverEuropebetween1940topresent:ProductUserGuide-Windstormtrack) for details).

Note that the `id` column in this table is defined separately for each algorithm.
For example, the windstorm with `id` 42 in the Hodges list is likely _not_ the same as the windstorm with that `id` in the TempestExtremes list.

This notebook uses [earthkit-data](https://github.com/ecmwf/earthkit-data) to download files from the CDS.
If you intend to run this notebook multiple times, it is highly recommended that you [enable caching](https://earthkit-data.readthedocs.io/en/latest/concepts/caching.html) to prevent having to download the same files multiple times.
If you prefer not to use earthkit, the following requests can also be used with the [cdsapi module](https://cds.climate.copernicus.eu/how-to-api).
In either case (earthkit-data or cdsapi), it is required to set up a CDS account and API key as explained [on the CDS website](https://cds.climate.copernicus.eu/how-to-api).

In [ ]:
# Define request for variables, based on CDS request form
dataset = "sis-european-wind-storm-reanalysis"

request = {
    "product": "windstorm_track",
    "variable": "all",
    "tracking_algorithm": ["hodges", "tempestextremes"],
    "event_aggregation": "single_event",
    "year":  [f"{year}"     for year  in range(year_start, year_end+1)],
    "month": [f"{month:02}" for month in range(1, 13)],
    "day":   [f"{day:02}"   for day   in range(1, 32)],
}

# Download data and load into Pandas
tracks = ekd.from_source("cds", dataset, request)
tracks = tracks.to_pandas()

# Pre-process: Use Pandas timestamps and set a custom index
tracks["time"] = pd.to_datetime(tracks["time"])
tracks = tracks.set_index(["algorithm", "id"])  # Reindex

# Display in notebook
tracks

#### Calculate windstorm duration
We calculate windstorm duration as simply the time difference between the first and last detections of a given windstorm ID:

In [ ]:
# Generate summary table
track_summary = summarise_track_dates(tracks)

# Compute track duration
track_summary = compute_track_durations(track_summary)

# Display in notebook
track_summary

#### Summary statistics
We quantify the distributions of windstorm durations through the mean, median, standard deviation, and 25th and 75th percentiles (first and third quartiles):

In [ ]:
# Change this to do all season, make that the first index level
stats = track_duration_statistics(track_summary)

# Display in notebook
display_track_duration_statistics(stats)

On average we find that the average lifetime for TempestExtremes is larger than that of Hodges, however there is larger variation (high standard deviation and IQR) in each to definitively say.

In [ ]:
# stats["DJF"].round(2)

This is also seen in the winter months, with the lifetime decreasing on average for Hodges whilst remaining more or less the same for Tempest. Quite a high amount of variation to definitely say though...

In [ ]:
# stats["JJA"].round(2)

Hard to definitely say again because of high variation.

#### Lifetime distribution
Next, we show the lifetime distribution of the two algorithms with a probability density function, binned at a 12 hour resolution (2x the temporal resolution).

In [ ]:
tracks_start_end = get_track_ends(track_data)
hodges_track_start_end = tracks_start_end[tracks_start_end["algorithm"] == "hodges"]
tempest_track_start_end = tracks_start_end[tracks_start_end["algorithm"] == "tempest_extremes"]

In [ ]:
plot_lifetime_distribution(
    data=[hodges_track_start_end, tempest_track_start_end],
    labels=["hodges", "tempestextremes"],
    colors = colors,
    bins=np.arange(0, 240, 12),
)

* Spread in TempestExtremes is larger.
* Peak of TempestExtremes longer.
* TempestExtremes finds ETCs that last longer.
* Hodges finds ETCs that are shorter.

#### Lifetime distribution (matchups only)
It is interesting to compare the distribution between each ETC in a matchup pair for the "same storm". We look at the probability density function, again binning at a 12 hour resolution (2x temporal resolution).

In [ ]:
# Separate start / end points based on algorithm.
h_track = tracks_start_end[tracks_start_end["algorithm"] == "hodges"]
te_track = tracks_start_end[tracks_start_end["algorithm"] == "tempest_extremes"]

# Filter only hodges id's that match.
h_ids_to_keep = filtered_matches_only["hodges_id"] 

h_filtered_tracks = h_track[
    h_track["id"].isin(h_ids_to_keep)
]

# Filter only tempest id's that match.
te_ids_to_keep = filtered_matches_only["tempest_id"]

te_filtered_tracks = te_track[
    te_track["id"].isin(te_ids_to_keep)
]

In [ ]:
# Plot lifetime distribution between match ups.
plot_lifetime_distribution(
    data=[h_filtered_tracks, te_filtered_tracks],
    labels=["hodges", "tempestextremes"],
    colors = colors,
    bins=np.arange(0, 240, 12),
)

The difference in lifetimes between Hodges and Tempest Extremes is also plotted as a histogram.

In [ ]:
diffs = []

for hodges_id, tempest_id in zip(filtered_matches_only["hodges_id"], filtered_matches_only["tempest_id"]):
    
    h_row = h_filtered_tracks[h_filtered_tracks["id"] == hodges_id].iloc[0]
    t_row = te_filtered_tracks[te_filtered_tracks["id"] == tempest_id].iloc[0]


    start_diff = (h_row["start_time"] - t_row["start_time"]).total_seconds()
    end_diff = (h_row["end_time"] - t_row["end_time"]).total_seconds()

    total_diff = (end_diff - start_diff)/(60*60)

    diffs.append(total_diff) 

# Histogram
hist, bin_edges = np.histogram(diffs, bins=np.arange(-240, 240, 12))

# Bin centres
centres = (bin_edges[:-1] + bin_edges[1:]) / 2

# Plot
plt.plot(centres, hist, label="Differences")

# Labels and styling
plt.xlabel("[Hodges - TE] Lifetime (hours)")
plt.ylabel("ETC Count")
plt.title("Lifetime  Difference Distribution")
plt.grid(True)
plt.legend()

plt.show()

* Peak at around zero.
* Roughly symmetrical.

**Not sure what information we can obtain from above**

## ℹ️ If you want to know more

### Key resources
The CDS catalogue entries for the data used were:
* Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present: [sis-european-wind-storm-reanalysis](https://doi.org/10.24381/bf1f06a9)

More about windstorm tracking algorithms:
* [IMILAST: A Community Effort to Intercompare Extratropical Cyclone Detection and Tracking Algorithms](https://doi.org/10.1175/BAMS-D-11-00154.1)
* [The numerous approaches to tracking extratropical cyclones and the challenges they present](https://doi.org/10.1002/wea.3861)
* Hodges / TRACK
  * [Feature Tracking on the Unit Sphere](https://doi.org/10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2)
  * [Adaptive Constraints for Feature Tracking](https://doi.org/10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2)
  * [New Perspectives on the Northern Hemisphere Winter Storm Tracks](https://doi.org/10.1175/1520-0469(2002)059<1041:NPOTNH>2.0.CO;2)
* TempestExtremes
  * [TempestExtremes v2.1: a community framework for feature detection, tracking, and analysis in large datasets](https://doi.org/10.5194/gmd-14-5023-2021)

Code libraries used:
* [GeoPandas](https://geopandas.org/en/stable/)
* [Haversine](https://github.com/mapado/haversine)
* [earthkit](https://earthkit.ecmwf.int/)
  * [earthkit-data](https://github.com/ecmwf/earthkit-data)
  * [earthkit-plots](https://github.com/ecmwf/earthkit-plots)

### References
[[Garnier+18](https://doi.org/10.1016/j.coastaleng.2017.06.014)] E. Garnier, P. Ciavola, T. Spencer, O. Ferreira, C. Armaroli, and A. McIvor, ‘Historical analysis of storm events: Case studies in France, England, Portugal and Italy’, Coastal Engineering, vol. 134, pp. 10–23, Apr. 2018, doi: 10.1016/j.coastaleng.2017.06.014.

[[Hersbach+20](https://doi.org/10.1002/qj.3803)] H. Hersbach et al., ‘The ERA5 global reanalysis’, Quarterly Journal of the Royal Meteorological Society, vol. 146, no. 730, pp. 1999–2049, May 2020, doi: 10.1002/qj.3803.

[[Hodges+95](https://doi.org/10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2)] K. I. Hodges, ‘Feature Tracking on the Unit Sphere’, Monthly Weather Review, vol. 123, no. 12, pp. 3458–3465, Dec. 1995, doi: 10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.&ZeroWidthSpace;CO;2.

[[Hodges+99](https://doi.org/10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2)] K. I. Hodges, ‘Adaptive Constraints for Feature Tracking’, Monthly Weather Review, vol. 127, no. 6, pp. 1362–1373, Jun. 1999, doi: 10.1175/1520-0493(1999)127<1362:ACFFT>2.0.&ZeroWidthSpace;CO;2.

[[Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)] K. I. Hodges, B. J. Hoskins, J. Boyle, and C. Thorncroft, ‘A Comparison of Recent Reanalysis Datasets Using Objective Feature Tracking: Storm Tracks and Tropical Easterly Waves’, Monthly Weather Review, vol. 131, no. 9, pp. 2012–2037, Sep. 2003, doi: 10.1175/1520-0493(2003)131<2012:ACORRD>2.0.&ZeroWidthSpace;CO;2.

[[Hoskins+02](https://doi.org/10.1175/1520-0469(2002)059<1041:NPOTNH>2.0.CO;2)] B. J. Hoskins and K. I. Hodges, ‘New Perspectives on the Northern Hemisphere Winter Storm Tracks’, Journal of the Atmospheric Sciences, vol. 59, no. 6, pp. 1041–1061, Mar. 2002, doi: 10.1175/1520-0469(2002)059<1041:NPOTNH>2.0.&ZeroWidthSpace;CO;2.

[[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)] U. Neu et al., ‘IMILAST: A Community Effort to Intercompare Extratropical Cyclone Detection and Tracking Algorithms’, Bulletin of the American Meteorological Society, vol. 94, no. 4, pp. 529–547, Apr. 2013, doi: 10.1175/BAMS-D-11-00154.1.

[[Ullrich+21](https://doi.org/10.5194/gmd-14-5023-2021)] P. A. Ullrich, C. M. Zarzycki, E. E. McClenny, M. C. Pinheiro, A. M. Stansfield, and K. A. Reed, ‘TempestExtremes v2.1: a community framework for feature detection, tracking, and analysis in large datasets’, Geoscientific Model Development, vol. 14, no. 8, pp. 5023–5048, Aug. 2021, doi: 10.5194/gmd-14-5023-2021.

[[Walker+20](https://doi.org/10.1002/wea.3861)] E. Walker, D. Mitchell, and W. Seviour, ‘The numerous approaches to tracking extratropical cyclones and the challenges they present’, Weather, vol. 75, no. 11, pp. 336–341, Nov. 2020, doi: 10.1002/wea.3861.
